# RAG com documentos do Código Brasileiro de Trânsito

## Bibliotecas

In [1]:
import os
import re
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.llms import Ollama

## Configuração

In [2]:
# Configurações
class Config:
    EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
    LLM_MODEL = "llama3.2:3b"
    TEMPERATURE = 0.5
    CHUNK_SIZE = 1000
    CHUNK_OVERLAP = 200
    PERSIST_DIR = "./chroma_db_pt_legal"
    PDF_PATHS = [
        r"M:\\tcc\\pdfs\\questionario\\Sinalização.pdf",
        r"M:\\tcc\\pdfs\\questionario\\Direção Defensiva.pdf",
        r"M:\\tcc\\pdfs\\questionario\\Segurança no Transporte crianças e gestantes.pdf",
        r"M:\\tcc\\pdfs\\questionario\\Noções de Primeiros Socorros no Trânsito.pdf",
        r"M:\\tcc\\pdfs\\questionario\\Manual de Primeiros Socorros.pdf"
    ]

In [3]:
# Função de limpeza
def clean_brazilian_legal_text(text: str) -> str:
    patterns = [
        r"Diário Oficial .+? Página \d+",
        r"Lei Nº \d+\.\d+ de \d{2}/\d{2}/\d{4}",
        r"Publicado em: \d{2}/\d{2}/\d{4}",
        r"Este texto não substitui o original publicado",
        r"\n\s*\d+\s*\n"
    ]
    
    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    
    text = re.sub(r"(?i)(artigo|art\.) ?(\d+)", r"Art. \2", text)
    text = re.sub(r"§ ?(único|\d+º?)", r"§ \1", text)
    
    return re.sub(r"\s+", " ", text).strip()

## Processamento de Documentos

In [4]:
# Carregar e processar documentos
documents = []
for path in Config.PDF_PATHS:
    try:
        loader = PyPDFLoader(path)
        pages = loader.load_and_split(text_splitter=None)
        for page in pages:
            cleaned = clean_brazilian_legal_text(page.page_content)
            if cleaned.strip():
                page.page_content = cleaned
                documents.append(page)
        print(f"{os.path.basename(path)} processado")
    except Exception as e:
        print(f"Erro em {path}: {str(e)}")

Sinalização.pdf processado
Direção Defensiva.pdf processado
Segurança no Transporte crianças e gestantes.pdf processado
Noções de Primeiros Socorros no Trânsito.pdf processado
Manual de Primeiros Socorros.pdf processado


## Divisão de Texto e Embeddings

### Chunks

In [5]:
# Configuração do text splitter jurídico corrigida
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=Config.CHUNK_SIZE,
    chunk_overlap=Config.CHUNK_OVERLAP,
    separators=[
        r"\n\nArt\. \d+\.",  
        r"\n§ ",             
        r"\n\n", 
        r"\n", 
        " ", 
        ""
    ],
    length_function=lambda x: len(x.split()),
    is_separator_regex=False  
)

In [6]:
# Processamento de documentos
texts = text_splitter.split_documents(documents)

# Verificação pós-split
print(f"Total de chunks gerados: {len(texts)}")
print("Exemplo de chunk inicial:")
print(texts[0].page_content[:500] + "...")

Total de chunks gerados: 320
Exemplo de chunk inicial:
DENATRAN...


In [7]:
# Verificar duplicatas
seen = set()
duplicates = 0
for t in texts:
    h = hash(t.page_content.strip().lower())
    if h in seen:
        duplicates += 1
    seen.add(h)
print(f"Chunks duplicados: {duplicates}")

Chunks duplicados: 0


### Embeddings

In [8]:
# Inicializar embeddings
embeddings = HuggingFaceEmbeddings(model_name=Config.EMBEDDING_MODEL)

C:\Users\mathe\AppData\Local\Temp\ipykernel_8032\2136887970.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=Config.EMBEDDING_MODEL)
c:\Users\mathe\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# Criar vetorstore
db = Chroma.from_documents(
    texts,
    embeddings,
    persist_directory=Config.PERSIST_DIR,
    collection_metadata={"hnsw:space": "cosine"}
)

In [10]:
# Configurar retriever
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 7,
        "lambda_mult": 0.45,
        "score_threshold": 0.25
    }
)

## Configuração do Modelo e Cadeia QA

In [11]:
# Inicializar LLM
llm = Ollama(
    model=Config.LLM_MODEL,
    temperature=Config.TEMPERATURE,
    system="Você é um especialista em legislação de trânsito brasileira. Responda com base nos documentos fornecidos."
)

C:\Users\mathe\AppData\Local\Temp\ipykernel_8032\2076523415.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(


In [12]:
# Prompt template
prompt_template = """
    Analise os seguintes trechos e responda de forma estruturada:
    
    Contexto:
    {context}
    
    Pergunta: {question}
    
    Inclua:
    1. Base legal (artigos/parágrafos)
    2. Explicação técnica
    3. Fontes (documento e página)
    
    Resposta:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [13]:
# Criar cadeia QA
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

## Função de Consulta

In [14]:
def consultar(pergunta):
    try:
        result = qa_chain({"query": pergunta})
        
        print(f"\nPergunta: {pergunta}")
        print(f"\nResposta:\n{result['result']}")
        
        print("\nFontes:")
        seen_sources = set()
        for doc in result['source_documents']:
            src = f"{os.path.basename(doc.metadata['source'])} - Página {doc.metadata['page']}"
            if src not in seen_sources:
                print(f"  - {src}")
                seen_sources.add(src)
                
    except Exception as e:
        print(f"Erro: {str(e)}")

## Exemplos de Uso

In [15]:
consultar("Qual é a idade mínima para habilitação na categoria D?")

C:\Users\mathe\AppData\Local\Temp\ipykernel_8032\172310940.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa_chain({"query": pergunta})



Pergunta: Qual é a idade mínima para habilitação na categoria D?

Resposta:
Infelizmente, não encontrei nenhuma informação sobre a idade mínima para habilitação na categoria D nos documentos fornecidos.

No entanto, posso sugerir uma pesquisa adicional para encontrar essa informação. De acordo com o Decreto nº 6.914/2009, que estabelece as normas para a habilitação de condutores, os requisitos para a habilitação de condutor de categoria D são:

* Idade mínima: 18 anos (Art. 12 do Decreto nº 6.914/2009)
* Exigências de saúde: ser capaz de realizar as tarefas necessárias à condução, sem comprometer a segurança do tráfego e da própria vida (Art. 13 do Decreto nº 6.914/2009)

É importante notar que essas informações podem estar disponíveis em outras normas ou regulamentos específicos, e não necessariamente no documento fornecido.

Se você tiver mais informações sobre a categoria D ou o contexto específico em que está procurando por essa informação, posso tentar ajudá-lo novamente.

Fontes

In [16]:
consultar("Quais as penalidades por dirigir sob efeito de álcool?")


Pergunta: Quais as penalidades por dirigir sob efeito de álcool?

Resposta:
**Resposta**

A pergunta referida ao artigo 44, item 14 do CTB, que trata sobre penalidades por dirigir sob efeito de álcool.

1. **Base legal (artigos/parágrafos)**: 
   - Art. 61 § 2º da Lei nº 9.394, de 4 de fevereiro de 1997, que regulamenta a infração prevista no artigo anterior, estabelece que "O dirigir sob o efeito de álcool ou drogas será punido com multa, desde que não haja morte ou lesão corporea grave; se houver morte ou lesão corporea grave, será punido com pena de detenção."
   - Art. 59 da Lei nº 9.394/97, que estabelece que "As infrações previstas no artigo anterior serão punidas pela Administração Tributária e pela Justiça Militar, conforme a legislação aplicável."
2. **Explicação técnica**: 
   - A penalidade de dirigir sob efeito de álcool é considerada uma infração grave, pois pode causar acidentes de trânsito graves, resultando em morte ou lesão corporea grave.
   - De acordo com o artigo 

In [17]:
consultar("Compare as exigências para as categorias B e D")


Pergunta: Compare as exigências para as categorias B e D

Resposta:
Aqui está a comparação das exigências para as categorias B e D de veículos, de acordo com o texto fornecido:

**Categorização de Veículos**

* **Categoria B**
	1. Base legal: Art. 60 do CTB (Código de Trânsito Brasileiro)
	2. Explicação técnica: A categoria B é destinada a veículos que transportam até 8 pessoas, incluindo o motorista.
	3. Fonte: Documento não especificado, mas pode ser encontrado no artigo 60 do CTB.
* **Categoria D**
	1. Base legal: Art. 60 do CTB (Código de Trânsito Brasileiro)
	2. Explicação técnica: A categoria D é destinada a veículos que transportam carga ou mercadorias, e não pode ser utilizada para transporte de pessoas.
	3. Fonte: Documento não especificado, mas pode ser encontrado no artigo 60 do CTB.

**Distância de Reserva (Dr)**

* **Categoria B**: Dr = Vf . 3,6 + 10
* **Categoria D**: Dr = Vf . 3,6 + 10 (mesmo valor)

**Velocidade Regulamentada Final (Vf)**

* **Categoria B**: Vf é a vel

## Respondendo Teste

In [18]:
consultar(
    '''
    Escolha a opção correta: Para habilitar-se na categoria “D”, o candidato deverá ser: 
    a) maior de 18 anos; 
    b) penalmente imputável, não importando sua idade; 
    c) menor de 21 anos; 
    d) maior de 21 anos"
    '''
)


Pergunta: 
    Escolha a opção correta: Para habilitar-se na categoria “D”, o candidato deverá ser: 
    a) maior de 18 anos; 
    b) penalmente imputável, não importando sua idade; 
    c) menor de 21 anos; 
    d) maior de 21 anos"
    

Resposta:
**Opção Correta:** a) maior de 18 anos.

**Base Legal:**
O artigo 60, parágrafo 1º do Código de Trânsito Brasileiro (CTB), estabelece que "A habilitação para dirigir veículos automotores é obrigatória para os cidadãos brasileiros maiores de 18 anos."

**Explicação Técnica:**
A categoria "D" é uma categoria de habilitação específica para veículos com motor mais potente, e sua obtenção é sujeita a requisitos específicos. De acordo com o CTB, os candidatos devem atender a um conjunto de critérios, incluindo ser maior de 18 anos.

**Fontes:**

* Código de Trânsito Brasileiro (CTB) - Artigo 60, parágrafo 1º.
* Portaria nº 476/2015 do Ministério da Infraestrutura de Transporte, que estabelece as regras para a habilitação de motoristas.

**Nota:*

In [19]:
consultar(
'''
    Escolha a opção correta: Para dirigir com segurança, evitando acidentes, o condutor deve demonstrar:
    
    Opção 1: habilidade ao dirigir; conhecimento das regras de trânsito; cooperação com os demais usuários da via;
    Opção 2: conhecimento de algumas regras de trânsito; agressividade nas situações perigosas; habilidade ao dirigir;
    Opção 3: bom senso; respeito apenas às regras mais importantes para a segurança; habilidade ao dirigir;
    Opção 4: habilidade ao dirigir; conhecimento de algumas regras de trânsito; bom senso.
    '''
)


Pergunta: 
    Escolha a opção correta: Para dirigir com segurança, evitando acidentes, o condutor deve demonstrar:
    
    Opção 1: habilidade ao dirigir; conhecimento das regras de trânsito; cooperação com os demais usuários da via;
    Opção 2: conhecimento de algumas regras de trânsito; agressividade nas situações perigosas; habilidade ao dirigir;
    Opção 3: bom senso; respeito apenas às regras mais importantes para a segurança; habilidade ao dirigir;
    Opção 4: habilidade ao dirigir; conhecimento de algumas regras de trânsito; bom senso.
    

Resposta:
A resposta correta é:

**Opção 4: habilidade ao dirigir; conhecimento de algumas regras de trânsito; bom senso.**

Base legal:

O Código de Trânsito Brasileiro (CTB) estabelece que o condutor deve demonstrar "bom senso" e conhecimento das regras de trânsito para dirigir com segurança (Artigo 27, § 3º, CTB).

Explicação técnica:

Para dirigir com segurança, é fundamental que o condutor demonstre habilidade ao dirigir, conhecim

In [20]:
consultar(
    '''
    Escolha a opção correta: O condutor de veículo deve dar preferência de passagem aos pedestres:
    
    Opção 1: somente quando estão atravessando na faixa de pedestres;
    Opção 2: que não tenham concluído a travessia, quando houver mudança de sinal;
    Opção 3: caso as pessoas estejam próximas a área escolar;
    Opção 4: somente quando isso for solicitado pelo agente de trânsito.
    '''
)


Pergunta: 
    Escolha a opção correta: O condutor de veículo deve dar preferência de passagem aos pedestres:
    
    Opção 1: somente quando estão atravessando na faixa de pedestres;
    Opção 2: que não tenham concluído a travessia, quando houver mudança de sinal;
    Opção 3: caso as pessoas estejam próximas a área escolar;
    Opção 4: somente quando isso for solicitado pelo agente de trânsito.
    

Resposta:
**Resposta**

A opção correta é:

Opção 2: que não tenham concluído a travessia, quando houver mudança de sinal.

**Base Legal**

O artigo 30 do Código de Trânsito Brasileiro (CTB) estabelece que "o condutor deve dar preferência de passagem aos pedestres" em determinadas situações. No entanto, não especifica quando essa preferência deve ser concedida.

O artigo 42, § 3º, do CTB estabelece que "se o pedestre não tiver concluído a travessia e houver mudança de sinal, o veículo deve dar preferência de passagem".

**Explicação Técnica**

A preferência de passagem aos pedestres 